In [1]:
%%capture
!pip -q install yfinance scikit-learn

In [6]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf

from io import StringIO
from contextlib import redirect_stdout, redirect_stderr
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor


# ============================================================
# Parameters
# ============================================================

ticker = "HTZ"
period = "10y"

h = 63                  # prediction horizon: approximately 3 months
alpha = 0.05            # 95% predictive region
M = 3                   # number of sigmoidal units; use M=2 or M=3

train_fraction = 0.70   # first 70% for training
random_state = 123


# ============================================================
# Silent download of adjusted daily prices
# ============================================================

buf = StringIO()

with redirect_stdout(buf), redirect_stderr(buf):
    raw = yf.download(
        ticker,
        period=period,
        auto_adjust=True,
        progress=False
    )

if raw.empty:
    raise RuntimeError("No data were downloaded.")

# Robust extraction of adjusted close
if isinstance(raw.columns, pd.MultiIndex):
    if "Close" in raw.columns.get_level_values(0):
        close = raw["Close"]
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
    elif "Close" in raw.columns.get_level_values(1):
        close = raw.xs("Close", axis=1, level=1).iloc[:, 0]
    else:
        raise RuntimeError("Could not find Close column.")
else:
    close = raw["Close"]

close = close.dropna()


# ============================================================
# Build data frame
# ============================================================

df = pd.DataFrame(index=close.index)
df["Close"] = close
df["logS"] = np.log(df["Close"])

# Daily log-return
df["r1"] = df["logS"].diff()

# Lagged cumulative log-returns
df["r5"] = df["logS"].diff(5)
df["r21"] = df["logS"].diff(21)
df["r63"] = df["logS"].diff(63)

# Realized volatility features
df["rv21"] = df["r1"].rolling(21).std()
df["rv63"] = df["r1"].rolling(63).std()
df["rv63_ann"] = np.sqrt(252) * df["rv63"]

# Moving-average gap features
df["ma21_gap"] = df["Close"] / df["Close"].rolling(21).mean() - 1.0
df["ma63_gap"] = df["Close"] / df["Close"].rolling(63).mean() - 1.0

# Target variable:
# 63-day future log-return
#
# Y_t = log(S_{t+h}/S_t)
df["Y"] = df["logS"].shift(-h) - df["logS"]


# ============================================================
# Feature matrix and target vector
# ============================================================

feature_cols = [
    "r1",
    "r5",
    "r21",
    "r63",
    "rv21",
    "rv63",
    "rv63_ann",
    "ma21_gap",
    "ma63_gap"
]

# Rows with known future return are used for training/calibration
labeled = df.dropna(subset=feature_cols + ["Y"]).copy()

# Latest row only needs features, not future Y
latest = df.dropna(subset=feature_cols).iloc[-1:].copy()

if len(labeled) < 200:
    raise RuntimeError("Not enough labeled observations after feature construction.")

X = labeled[feature_cols].to_numpy()
Y = labeled["Y"].to_numpy()

X_latest = latest[feature_cols].to_numpy()
S_latest = float(latest["Close"].iloc[0])


# ============================================================
# Chronological split:
# training window + calibration/backtesting window
# ============================================================

n = len(labeled)
n_train = int(train_fraction * n)

X_train = X[:n_train]
Y_train = Y[:n_train]

X_cal = X[n_train:]
Y_cal = Y[n_train:]

if len(X_cal) < 50:
    raise RuntimeError("Calibration window is too small.")


# ============================================================
# Sigmoidal approximation of F
#
# F_hat(z) = c + sum_{j=1}^M alpha_j sigma(w_j'z + b_j)
#
# Implemented as a one-hidden-layer MLP with logistic activation.
# ============================================================

model = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        hidden_layer_sizes=(M,),
        activation="logistic",
        solver="lbfgs",
        alpha=1e-4,
        max_iter=5000,
        random_state=random_state
    ))
])

model.fit(X_train, Y_train)


# ============================================================
# Residual calibration
# ============================================================

Y_cal_hat = model.predict(X_cal)

residuals = Y_cal - Y_cal_hat
absolute_residuals = np.abs(residuals)

q = float(np.quantile(absolute_residuals, 1.0 - alpha))


# ============================================================
# Latest predictive region G_alpha
#
# If F_hat predicts the 63-day log-return, then:
#
# G_alpha(S_t, z_t)
# =
# [
#   S_t exp(F_hat(z_t) - q),
#   S_t exp(F_hat(z_t) + q)
# ]
# ============================================================

F_hat_latest = float(model.predict(X_latest)[0])

G_lower = S_latest * np.exp(F_hat_latest - q)
G_upper = S_latest * np.exp(F_hat_latest + q)

print(f"[{G_lower:.2f}, {G_upper:.2f}]")

[1.96, 11.44]
